# AskMeDB - Natural Language Database Query Agent

This notebook demonstrates how to use **AskMeDB** to ask questions about a database in plain English.

AskMeDB converts your questions into SQL, executes them, self-corrects on errors, and gives you human-readable answers.

**What this notebook covers:**
1. Install dependencies
2. Set up a sample SQLite database (CloudMetrics SaaS data)
3. Upload knowledge files (schema, business rules, query patterns)
4. Ask questions in natural language

## 1. Install Dependencies

In [ ]:
!pip install git+https://github.com/manaranjanp/askmedb.git faker python-dotenv

## 2. Set Your API Key

AskMeDB uses LiteLLM under the hood, so you can use any supported provider.
Set the API key for your preferred LLM provider below.

In [ ]:
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Add your key in the Colab sidebar under "Secrets" (key icon)
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Loaded GROQ_API_KEY from Colab Secrets.")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    pass

# Option 2: Set directly (uncomment and paste your key)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Choose your model (default: Claude Haiku)
LLM_MODEL = os.environ.get("LLM_MODEL", "groq/llama-3.3-70b-versatile")
print(f"Using model: {LLM_MODEL}")

## 3. Create the Sample Database

We'll clone the repo to get `setup_db.py` and the knowledge files, then generate a sample CloudMetrics database with 5 tables and realistic synthetic data.

In [ ]:
!git clone https://github.com/manaranjanp/askmedb.git askmedb_repo

In [ ]:
!python askmedb_repo/examples/setup_db.py

In [ ]:
import sqlite3

DB_PATH = "askmedb_repo/examples/cloudmetrics.db"

# Quick check: list tables and row counts
conn = sqlite3.connect(DB_PATH)
cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
tables = [row[0] for row in cursor.fetchall()]

print("Tables in the database:")
for table in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")

conn.close()

## 4. Quick Start — Auto Schema Detection

The simplest way to use AskMeDB: point it at a database and let it auto-detect the schema.

In [ ]:
from askmedb import AskMeDBEngine, AskMeDBConfig, SQLiteConnector, AutoSchemaProvider

db = SQLiteConnector(DB_PATH)
schema = AutoSchemaProvider(db, database_name="cloudmetrics", description="SaaS analytics database")

engine = AskMeDBEngine(
    db=db,
    schema=schema,
    config=AskMeDBConfig(model=LLM_MODEL),
)

result = engine.ask("How many customers do we have?")

print(f"Question: {result.question}")
print(f"Reasoning:   {result.reasoning}")
print(f"SQL:      {result.sql}")
print(f"Answer:   {result.answer}")

## 5. Full Setup — With Knowledge Files

For better results, provide AskMeDB with business context: schema descriptions, metric definitions, and SQL patterns.

The knowledge files are included in the cloned repo at `examples/cloudmetrics/knowledge/`.

In [ ]:
KNOWLEDGE_DIR = "askmedb_repo/examples/cloudmetrics/knowledge"

# Show what knowledge files are available
!ls -la {KNOWLEDGE_DIR}

In [ ]:
from askmedb import AskMeDBEngine, AskMeDBConfig, SQLiteConnector, JSONSchemaProvider
import os

config = AskMeDBConfig(
    model=LLM_MODEL,
    max_correction_attempts=3,
    enable_learnings=True,
    learnings_path=os.path.join(KNOWLEDGE_DIR, "learnings.json"),
)

engine = AskMeDBEngine(
    db=SQLiteConnector(DB_PATH),
    schema=JSONSchemaProvider(os.path.join(KNOWLEDGE_DIR, "schema.json")),
    config=config,
    business_rules=os.path.join(KNOWLEDGE_DIR, "business_rules.json"),
    query_patterns=os.path.join(KNOWLEDGE_DIR, "query_patterns.sql"),
    agent_description=(
        "You are a data analyst agent for CloudMetrics, a B2B SaaS company "
        "that sells project management software. "
        "You answer natural language questions by generating SQL queries."
    ),
)

print("Engine ready with full knowledge context!")

## 6. Ask Questions

Now let's ask some business questions. Each `result` contains the generated SQL, raw data, and a human-readable answer.

In [ ]:
# Simple question
result = engine.ask("What are our plan names and prices?")

print(f"SQL: {result.sql}\n")
print(result.answer)

In [ ]:
# Business metric
result = engine.ask("What is our total MRR right now?")

print(f"SQL: {result.sql}\n")
print(result.answer)

In [ ]:
# Multi-turn follow-up (references previous context)
result = engine.ask("Break that down by plan")

print(f"SQL: {result.sql}\n")
print(result.answer)

In [ ]:
# Reset conversation before switching topics
engine.reset_conversation()

# Advanced query with joins and date logic
result = engine.ask("What is the average support ticket resolution time in hours, broken down by priority?")

print(f"SQL: {result.sql}\n")
print(result.answer)

In [ ]:
# Complex multi-table query
engine.reset_conversation()

result = engine.ask("Show me the top 5 customers by total revenue who have also filed more than 3 support tickets")

print(f"SQL: {result.sql}\n")
print(result.answer)
print(f"\nCorrection attempts: {result.correction_attempts}")
if result.warnings:
    print(f"Warnings: {result.warnings}")

## 7. Working with Results

The `QueryResult` object gives you structured access to all data.

In [ ]:
engine.reset_conversation()

result = engine.ask("Show me the number of customers by industry")

# Convert to pandas DataFrame
df = result.to_dataframe()
df

In [ ]:
# Convert to list of dicts
records = result.to_dicts()
records

In [ ]:
# Access raw data
print(f"Columns: {result.columns}")
print(f"Row count: {result.row_count}")
print(f"Success: {result.success}")

## 8. Event Hooks — See What's Happening

You can attach callbacks to monitor the full pipeline: reasoning, SQL generation, corrections, and more.

In [ ]:
engine.reset_conversation()

# Attach hooks to see the pipeline in action
engine.on_reasoning = lambda r: print(f"\n--- Reasoning ---\n{r}")
engine.on_sql_generated = lambda sql: print(f"\n--- Generated SQL ---\n{sql}")
engine.on_sql_error = lambda err, sql, attempt: print(f"\n!!! SQL Error (attempt {attempt}): {err}")
engine.on_sql_corrected = lambda sql, reason: print(f"\n--- Corrected ---\n{reason}\n{sql}")
engine.on_results = lambda cols, rows: print(f"\n--- Results: {len(rows)} rows ---")
engine.on_warning = lambda w: print(f"\n!!! Warning: {w}")

result = engine.ask("What is the monthly churn rate over the past 6 months?")

print(f"\n--- Answer ---\n{result.answer}")

## 9. Try Your Own Questions

The CloudMetrics database has 5 tables: `customers`, `plans`, `subscriptions`, `invoices`, `support_tickets`.

Try asking anything about SaaS metrics, customer data, billing, or support!

In [ ]:
# Clear hooks for cleaner output
engine.on_reasoning = None
engine.on_sql_generated = None
engine.on_sql_error = None
engine.on_sql_corrected = None
engine.on_results = None
engine.on_warning = None
engine.reset_conversation()

# Try your own question!
result = engine.ask("Which country has the most enterprise customers?")

print(f"SQL: {result.sql}\n")
print(result.answer)